In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.steam_reviews.reviews_raw")

df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

df.select("helpful_votes").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()


+-------+-----------+-----------+-------------+
|game_id|review_text|recommended|helpful_votes|
+-------+-----------+-----------+-------------+
|      0|       7305|          0|            0|
+-------+-----------+-----------+-------------+

+-------+-------------------+
|summary|      helpful_votes|
+-------+-------------------+
|  count|            6417106|
|   mean|0.14724456787841747|
| stddev|0.35434957975949816|
|    min|                  0|
|    25%|                  0|
|    50%|                  0|
|    75%|                  0|
|    max|                  1|
+-------+-------------------+



In [0]:
from pyspark.sql import functions as F

features = df.select(
    "helpful_votes",
    "recommended",
    F.length("review_text").alias("review_length"),
    F.size(F.split("review_text", r"\s+")).alias("word_count"),
)

features.groupBy("helpful_votes").agg(
    F.count("*").alias("n"),
    F.avg("review_length").alias("avg_length"),
    F.avg("word_count").alias("avg_words"),
).show()


+-------------+-------+-----------------+-----------------+
|helpful_votes|      n|       avg_length|        avg_words|
+-------------+-------+-----------------+-----------------+
|            0|5472222|292.5695424049229|53.58031718336372|
|            1| 944884|373.0260936317096|67.70455452673264|
+-------------+-------+-----------------+-----------------+



In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler

features = df.select(
    "helpful_votes",
    "recommended",
    F.length("review_text").alias("review_length"),
    F.size(F.split("review_text", r"\s+")).alias("word_count"),
    F.length(F.regexp_extract("review_text", r"(!)", 0)).alias("has_exclaim"),
    F.length(F.regexp_extract("review_text", r"(\?)", 0)).alias("has_question"),
).na.fill(0)

assembler = VectorAssembler(
    inputCols=["recommended", "review_length", "word_count", "has_exclaim", "has_question"],
    outputCol="features",
)
data = assembler.transform(features).select("features", F.col("helpful_votes").alias("label"))

train, test = data.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,}  Test: {test.count():,}")


Train: 5,132,618  Test: 1,284,488


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train)
lr_preds = lr_model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
print(f"Baseline Logistic Regression AUC: {evaluator.evaluate(lr_preds):.4f}")


Baseline Logistic Regression AUC: 0.5782


In [0]:
from pyspark.ml.feature import Tokenizer, HashingTF, IDF

tokenizer = Tokenizer(inputCol="review_text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=2000)
idf = IDF(inputCol="raw_features", outputCol="tfidf")

tokenized = tokenizer.transform(df.na.fill({"review_text": ""}))
tf = hashingTF.transform(tokenized)
idf_model = idf.fit(tf)
tfidf_data = idf_model.transform(tf)

tfidf_data.select("helpful_votes", "tfidf").show(3, truncate=60)


+-------------+------------------------------------------------------------+
|helpful_votes|                                                       tfidf|
+-------------+------------------------------------------------------------+
|            0|(2000,[764,1535,1568],[3.3468958881247337,3.8455235104902...|
|            1|(2000,[15,17,35,72,77,80,95,157,162,166,173,187,196,202,2...|
|            0|(2000,[317,488,975,1373,1568],[4.908205934164963,0.890312...|
+-------------+------------------------------------------------------------+
only showing top 3 rows


In [0]:
from pyspark.ml.feature import VectorAssembler

combined = tfidf_data.select(
    "helpful_votes",
    "recommended",
    F.length("review_text").alias("review_length"),
    F.size(F.split("review_text", r"\s+")).alias("word_count"),
    F.length(F.regexp_extract("review_text", r"(!)", 0)).alias("has_exclaim"),
    F.length(F.regexp_extract("review_text", r"(\?)", 0)).alias("has_question"),
    "tfidf",
).na.fill(0)

assembler2 = VectorAssembler(
    inputCols=["recommended", "review_length", "word_count", "has_exclaim", "has_question", "tfidf"],
    outputCol="features",
)
data2 = assembler2.transform(combined).select("features", F.col("helpful_votes").alias("label"))

train2, test2 = data2.randomSplit([0.8, 0.2], seed=42)

lr2 = LogisticRegression(featuresCol="features", labelCol="label")
lr2_model = lr2.fit(train2)
preds2 = lr2_model.transform(test2)

print(f"Logistic Regression + TF-IDF AUC: {evaluator.evaluate(preds2):.4f}")


Logistic Regression + TF-IDF AUC: 0.6572


In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/steam_reviews/raw_data/tmp"

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = (ParamGridBuilder()
    .addGrid(lr2.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr2.elasticNetParam, [0.0, 0.5])
    .build())

cv = CrossValidator(
    estimator=lr2,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
)

cv_model = cv.fit(train2)
best_preds = cv_model.bestModel.transform(test2)
print(f"Tuned AUC: {evaluator.evaluate(best_preds):.4f}")
print(f"Best regParam: {cv_model.bestModel.getRegParam()}, elasticNet: {cv_model.bestModel.getElasticNetParam()}")


Tuned AUC: 0.6576
Best regParam: 0.01, elasticNet: 0.0


In [0]:
# Confusion matrix
best_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# Coefficients for the 5 structured features (indices 0-4, before the TF-IDF block)
coefs = cv_model.bestModel.coefficients
structured_names = ["recommended", "review_length", "word_count", "has_exclaim", "has_question"]
for name, coef in zip(structured_names, coefs[:5]):
    print(f"{name}: {coef:.4f}")


+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0|1092278|
|    0|       1.0|   2919|
|    1|       0.0| 186966|
|    1|       1.0|   2325|
+-----+----------+-------+

recommended: -0.3241
review_length: 0.0000
word_count: -0.0000
has_exclaim: -0.0882
has_question: 0.2196


In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=False, withStd=True)
scaler_model = scaler.fit(train2)
train_scaled = scaler_model.transform(train2)
test_scaled = scaler_model.transform(test2)

lr3 = LogisticRegression(featuresCol="scaled_features", labelCol="label", regParam=0.01, elasticNetParam=0.0)
lr3_model = lr3.fit(train_scaled)
preds3 = lr3_model.transform(test_scaled)
print(f"Scaled AUC: {evaluator.evaluate(preds3):.4f}")

for name, coef in zip(structured_names, lr3_model.coefficients[:5]):
    print(f"{name}: {coef:.4f}")


Scaled AUC: 0.6576
recommended: -0.2492
review_length: 0.0097
word_count: -0.0024
has_exclaim: -0.0336
has_question: 0.0577


In [0]:
import json

results = {
    "n_reviews": 6417106,
    "class_balance": {"not_helpful": 5472222, "helpful": 944884},
    "auc": {"baseline": 0.5783, "tfidf": 0.6572, "tuned": 0.6576, "scaled": 0.6576},
    "best_hyperparams": {"regParam": 0.01, "elasticNetParam": 0.0},
    "confusion_matrix": {"tn": 1092278, "fp": 2919, "fn": 186966, "tp": 2325},
    "coefficients": {
        "recommended": -0.2492, "review_length": 0.0097, "word_count": -0.0024,
        "has_exclaim": -0.0336, "has_question": 0.0577
    },
}
with open("/Volumes/workspace/steam_reviews/raw_data/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("saved")


saved
